# 🗂️ Week 2 Assignment — Employee Data Cleaner

**Course:** Python Programming — Five-Week Foundations  
**Week:** 2 — File I/O · CSV Parsing · Data Cleaning · Summary Statistics  
**Estimated time:** 2–3 hours  
**Total points:** 100  

---

## Scenario

You've been handed `employees.csv` — an export from your company's HR system. Unfortunately it has the usual real-world problems: missing values, a bad salary entry, duplicate rows, and a missing email. Your job is to read the file, identify every problem, clean the data, compute summary statistics, and write a clean output file.

**How to use this notebook:**
- Read each Markdown cell carefully — it explains the concept before you use it
- Write your code in the cell directly below
- Run each cell with **Shift+Return**
- Do not edit the final **Self-check** cell

---

## Reference — Escape Characters

An **escape character** is a backslash `\` followed by a letter that represents something you can't easily type directly in a string — like a new line or a tab.

| Escape | Name | What it does |
|--------|------|--------------|
| `\n` | Newline | Moves to the next line |
| `\t` | Tab | Inserts a tab (large horizontal space) |
| `\\` | Backslash | Inserts a literal `\` character |
| `\'` | Single quote | Inserts `'` inside a single-quoted string |
| `\"` | Double quote | Inserts `"` inside a double-quoted string |

```python
# \n — newline
print("Name: Alice\nAge: 31\nCity: Chicago")
# Output:
# Name: Alice
# Age: 31
# City: Chicago

# \t — tab (useful for aligning columns)
print("Name\t\tSalary")
print("Alice\t\t$95,000")
print("Bob\t\t$62,000")
# Output:
# Name		Salary
# Alice		$95,000
# Bob		$62,000

# Escaping quotes inside a string
print("She said \"hello\".")
print('It\'s a great day.')

# \\ — a literal backslash
print("File path: C:\\Users\\Alice")
```

You'll use `\n` often this week to add blank lines between sections of output, and `\t` to align printed columns.

---

## Reference — The `with` Statement

Whenever you open a file in Python, you also need to close it when you're done. Forgetting to close a file can cause data loss or lock the file. The `with` statement handles this automatically — it **closes the file for you** the moment the indented block finishes, even if an error occurs.

```python
# Without with — you must remember to close manually
file = open("employees.csv", "r")
contents = file.read()
file.close()   # easy to forget!

# With 'with' — closes automatically, always
with open("employees.csv", "r") as file:
    contents = file.read()
# file is closed here — you don't need to do anything
```

The pattern is always:

```python
with open("filename", "mode") as variable_name:
    # everything indented here has access to the file
    # the file closes when this block ends
```

### File modes

| Mode | Meaning |
|------|---------|
| `"r"` | Read — opens an existing file for reading (default) |
| `"w"` | Write — creates a new file, or **overwrites** if it already exists |
| `"a"` | Append — adds to the end of an existing file without erasing it |

```python
# Reading a CSV line by line
with open("employees.csv", "r") as file:
    for line in file:
        print(line.strip())   # .strip() removes the \n at the end of each line

# Writing a new file
with open("report.txt", "w") as file:
    file.write("Summary Report\n")
    file.write("Total employees: 25\n")
```

> 💡 **Always use `with open(...)` instead of `open()` alone.** It is safer, shorter, and the standard Python style.

---

## Reference — `try` / `except`

Some operations can fail at runtime — for example, converting the string `"not_a_number"` to a float. Without protection, this crashes your entire program. `try` / `except` lets you **catch the error gracefully** and decide what to do instead.

```python
# Without try/except — crashes on bad input
salary = float("not_a_number")   # ValueError: could not convert string to float
```

```python
# With try/except — handles the error without crashing
value = "not_a_number"

try:
    salary = float(value)
    print(f"Salary is ${salary:,.2f}")
except ValueError:
    print(f"Could not convert '{value}' to a number — skipping this row.")
```

### How it works

1. Python runs the code inside `try`
2. If no error occurs — great, it continues normally
3. If the specified error occurs — Python jumps to `except` and runs that block instead
4. Either way, the program keeps running

### Common error types you'll encounter

| Error | When it happens |
|-------|-----------------|
| `ValueError` | Converting a string to a number fails (`int("abc")`) |
| `FileNotFoundError` | The file path you gave `open()` doesn't exist |
| `KeyError` | A dictionary key you asked for doesn't exist |
| `ZeroDivisionError` | You divided by zero |

```python
# Catching a missing file
try:
    with open("employees.csv", "r") as file:
        contents = file.read()
except FileNotFoundError:
    print("Error: employees.csv not found. Make sure the file is in the same folder as this notebook.")

# Catching multiple different errors
try:
    salary = float(row["salary"])
    age    = int(row["age"])
except ValueError as e:
    print(f"Bad data in row: {e}")
except KeyError as e:
    print(f"Missing column: {e}")
```

> 💡 The `as e` part captures the error message so you can print it. It's optional but very useful for debugging.

---

## Task 1 — Explore the Raw File
**15 points**

Before cleaning any data, always look at it first. Read `employees.csv` and print what you find.

**Requirements:**
1. Use `with open(...)` to open `employees.csv` in read mode.
2. Read and print the **header row** (the first line) by itself.
3. Print the **total number of data rows** (not counting the header).
4. Print the **first 3 data rows** in full so you can see what the data looks like.
5. Wrap the entire file-open block in a `try` / `except FileNotFoundError` so a helpful message prints if the file is missing.

> 💡 **Hint:** `csv.DictReader` automatically uses the first row as the header. Call `next(reader)` to grab just the header, or just start looping and each `row` will be a dictionary keyed by column name.

In [1]:
# Task 1 — Explore the raw file
import csv

try:
    with open("employees.csv", "r") as file:
        reader = csv.DictReader(file)

        # 2. Print the header (column names)
        print("Columns:", reader.fieldnames)

        # 3 & 4. Count rows and print the first 3
        row_count = 0
        for row in reader:
            row_count += 1
            if row_count <= 3:
                print(row)

        print(f"\nTotal data rows: {row_count}")
except FileNotFoundError:
    print("Error: employees.csv not found. Make sure it is in the same folder as this notebook.")

Columns: ['name', 'age', 'department', 'city', 'salary', 'start_date', 'email']
{'name': 'Alice Nguyen', 'age': '31', 'department': 'Engineering', 'city': 'San Francisco', 'salary': '95000', 'start_date': '2019-03-12', 'email': 'alice.nguyen@acmecorp.com'}
{'name': 'Bob Martinez', 'age': '27', 'department': 'Marketing', 'city': 'Chicago', 'salary': '62000', 'start_date': '2021-07-01', 'email': 'bob.martinez@acmecorp.com'}
{'name': 'Carol Kim', 'age': '35', 'department': 'Engineering', 'city': 'San Francisco', 'salary': '112000', 'start_date': '2017-11-20', 'email': 'carol.kim@acmecorp.com'}

Total data rows: 29


---
## Task 2 — Identify Data Problems
**20 points**

Read through every row and report each type of problem you find. Do **not** clean anything yet — just detect and report.

The file contains these problems for you to find:
- Missing age (blank field)
- Missing salary (blank field)
- A salary value that is not a valid number
- A missing email address
- Duplicate rows (same employee appearing more than once)

**Requirements:**
1. Loop through every row using `csv.DictReader`.
2. Check each row for the problems listed above and print a message for each one found. Example: `"⚠ Missing age: Dave Okafor"`
3. Use `try` / `except ValueError` to detect salary values that cannot be converted to a float.
4. Track duplicate rows using a set of names you've already seen.
5. At the end, print a summary count of each type of problem found.

> 💡 **Hint:** To detect duplicates, create an empty set called `seen_names` before the loop. Inside the loop, check `if row["name"] in seen_names` — if true, it's a duplicate. If not, add it with `.add()`.

In [2]:
# Task 2 — Identify data problems

missing_age_count    = 0
missing_salary_count = 0
bad_salary_count     = 0
missing_email_count  = 0
duplicate_count      = 0
seen_names           = set()

print("--- Data Problems Found ---\n")

with open("employees.csv", "r") as file:
    reader = csv.DictReader(file)
    for row in reader:
        name = row["name"]

        # Check for missing age
        if not row.get("age") or row.get("age").strip() == "":
            print(f"⚠ Missing age: {name}")
            missing_age_count += 1

        # Check for missing or invalid salary
        if not row.get("salary") or row.get("salary").strip() == "":
            print(f"⚠ Missing salary: {name}")
            missing_salary_count += 1
        else:
            try:
                float(row["salary"])
            except ValueError:
                print(f"⚠ Invalid salary value: {name} (salary = {row['salary']})")
                bad_salary_count += 1

        # Check for missing email
        if not row.get("email") or row.get("email").strip() == "":
            print(f"⚠ Missing email: {name}")
            missing_email_count += 1

        # Check for duplicates using seen_names set
        if name in seen_names and name != "":
            print(f"⚠ Duplicate row found: {name}")
            duplicate_count += 1
        else:
            seen_names.add(name)


print(f"\n--- Problem Summary ---")
print(f"Missing age:      {missing_age_count}")
print(f"Missing salary:   {missing_salary_count}")
print(f"Bad salary value: {bad_salary_count}")
print(f"Missing email:    {missing_email_count}")
print(f"Duplicate rows:   {duplicate_count}")

--- Data Problems Found ---

⚠ Missing age: Dave Okafor
⚠ Missing age: Nina Johansson
⚠ Missing salary: Nina Johansson
⚠ Missing email: Sam Nkosi
⚠ Invalid salary value: Tom Erikson (salary = not_a_number)
⚠ Duplicate row found: Alice Nguyen
⚠ Duplicate row found: Nina Johansson
⚠ Duplicate row found: Sam Nkosi

--- Problem Summary ---
Missing age:      2
Missing salary:   1
Bad salary value: 1
Missing email:    1
Duplicate rows:   3


---
## Task 3 — Clean the Data
**25 points**

Now apply the cleaning rules and build a list of valid rows.

**Cleaning rules:**
- **Skip** any row where `salary` is missing or cannot be converted to a float
- **Skip** any row where `name` is blank
- **Skip** duplicate rows — keep only the first occurrence of each name
- **Convert** `salary` from a string to a `float`
- **Convert** `age` from a string to an `int` where present; set it to `None` if blank
- **Strip** any leading or trailing whitespace from all string fields using `.strip()`

**Requirements:**
1. Loop through the file and apply all the rules above.
2. Append each valid, cleaned row (as a dictionary) to a list called `clean_rows`.
3. Print how many rows were kept and how many were skipped.

> 💡 **Hint:** Use `continue` inside the loop to skip a row early. When you hit a problem, print a message and then `continue` — Python will jump straight to the next row.

In [3]:
# Task 3 — Clean the data

clean_rows = []
skipped = 0
seen_names = set()

print("🧹 Cleaning data...\n")

with open("employees.csv", "r", newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    
    for row in reader:
        # Strip all fields
        row = {k: v.strip() if v else "" for k, v in row.items()}
        
        name = row["name"]
        
        if not name:
            skipped += 1
            continue
        
        if name in seen_names:
            skipped += 1
            continue
        
        if not row.get("salary"):
            skipped += 1
            continue
        
        try:
            row["salary"] = float(row["salary"])
        except ValueError:
            skipped += 1
            continue
        
        # Convert age
        if row.get("age"):
            try:
                row["age"] = int(row["age"])
            except ValueError:
                row["age"] = None
        else:
            row["age"] = None
        
        seen_names.add(name)
        clean_rows.append(row)

print(f"Rows kept: {len(clean_rows)}")
print(f"Rows skipped: {skipped}")

🧹 Cleaning data...

Rows kept: 25
Rows skipped: 4


---
## Task 4 — Summary Statistics
**20 points**

Use the cleaned data to answer some questions about the workforce.

**Requirements:**
1. Calculate and print the **minimum**, **maximum**, and **average** salary across all clean rows. Format salary values with `$` and commas: `$95,000.00`
2. Print the **headcount per department** (how many employees are in each department).
3. Print the **average salary per city**, sorted from highest to lowest.
4. Print the **names of the top 3 highest-paid employees**.

> 💡 **Hint:** Use a dictionary to group rows by department or city. The keys are department/city names and the values are lists of salaries. Then loop over the dictionary to compute averages.
>
> For sorting, `sorted(my_dict.items(), key=lambda x: x[1], reverse=True)` sorts a dictionary by value from highest to lowest.

In [4]:
# Task 4 — Summary statistics

# 1. Min, max, average salary
salaries = [row["salary"] for row in clean_rows]
min_salaries = min(salaries)
max_salaries = max(salaries)
average_salary = sum(salaries) / len(salaries)

print("--- Salary Summary ---")
print(f"Minimum: ${min_salaries:>10,.2f}")
print(f"Maximum:  ${max_salaries:>10,.2f}")
print(f"Average:  ${average_salary:>10,.2f}")

# 2. Headcount per department
print("\n--- Headcount by Department ---")
dept_count = {}
# Your code here
for row in clean_rows:
    dept = row["department"]
    if dept in dept_count:
        dept_count[dept] += 1
    else:
        dept_count[dept] = 1

for dept, count in dept_count.items():
    print(f"{dept}: {count}")

# 3. Average salary by city, sorted high to low
print("\n--- Average Salary by City ---")
city_salaries = {}
# Your code here
for city, salaries in city_salaries.items():
    city_average = sum(salaries) / len(salaries)
    print(f"{city}: Average Salary: ${city_average:.2f}")

# 4. Top 3 highest-paid employees
print("\n--- Top 3 Earners ---")
# Your code here
top_earners = sorted(clean_rows, key=lambda x: x["salary"], reverse=True)[:3]
for earner in top_earners:
    print(f"{earner['name']}: ${earner['salary']:.2f}")

--- Salary Summary ---
Minimum: $ 47,000.00
Maximum:  $125,000.00
Average:  $ 80,500.00

--- Headcount by Department ---
Engineering: 9
Marketing: 6
Sales: 6
HR: 4

--- Average Salary by City ---

--- Top 3 Earners ---
Victor Alves: $125000.00
Marcus Reed: $119000.00
Carol Kim: $112000.00


---
## Task 5 — Write the Cleaned File
**10 points**

Write your cleaned rows to a new file called `employees_clean.csv` using `csv.DictWriter`.

**Requirements:**
1. Open `employees_clean.csv` in write mode using `with open(...)`.
2. Use `csv.DictWriter` with the correct `fieldnames`.
3. Write the header row using `.writeheader()`.
4. Write all rows from `clean_rows` using `.writerows()`.
5. Print a confirmation message showing the filename and how many rows were written.

> 💡 **Hint:** When opening a CSV for writing on Mac, pass `newline=""` as an argument to `open()` — this prevents Python from adding extra blank lines between rows:
> ```python
> with open("employees_clean.csv", "w", newline="") as file:
> ```

In [5]:
# Task 5 — Write the cleaned file

fieldnames = ["name", "age", "department", "city", "salary", "start_date", "email"]

with open("employees_clean.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames, extrasaction="ignore")
    # Your code here
    writer.writeheader()
    for row in clean_rows:
        writer.writerow(row)

print(f"Wrote {len(clean_rows)} rows to employees_clean.csv")

Wrote 25 rows to employees_clean.csv


---
## Task 6 — Commit to GitHub
**10 points**

Save your work and push it to your course repository.

**Steps:**
1. Save this notebook: **File → Save Notebook** (`Cmd+S`).
2. Make sure `employees.csv` and `employees_clean.csv` are in the `week2/` folder.
3. Open Terminal and run:

```bash
cd ~/Documents/python-course
git add .
git commit -m "Week 2 assignment: employee data cleaner"
git push
```

4. Verify all three files (`week2_assignment.ipynb`, `employees.csv`, `employees_clean.csv`) appear in the `week2/` folder on github.com.

---
## Grading Rubric

| Task | Points | Full marks when… |
|------|--------|-------------------|
| 1 — Explore | 15 | Header and row count print correctly; `FileNotFoundError` handled |
| 2 — Identify | 20 | All 5 problem types detected and counted correctly |
| 3 — Clean | 25 | Correct rows skipped; types converted; clean_rows populated |
| 4 — Statistics | 20 | All four outputs correct and formatted with `$` and commas |
| 5 — Write | 10 | `employees_clean.csv` written with correct header and rows |
| 6 — GitHub | 10 | All three files committed to `week2/` with a clear message |
| **Total** | **100** | |

---
## Before You Submit — Checklist

- [ ] All cells run top-to-bottom without errors (**Kernel → Restart & Run All**)
- [ ] Task 2 finds exactly: 2 missing ages, 1 missing salary, 1 bad salary, 1 missing email, 3 duplicates
- [ ] Task 3 keeps 25 clean rows and skips 4
- [ ] `employees_clean.csv` exists in the `week2/` folder
- [ ] All salary output is formatted with `$` and commas
- [ ] All three files are committed and visible on GitHub

In [6]:
# Self-check — DO NOT EDIT THIS CELL
import os

print("=" * 45)
print("  Week 2 Self-check")
print("=" * 45)

checks = []

# Check 1: clean_rows exists and is a list
try:
    assert isinstance(clean_rows, list), "clean_rows should be a list"
    assert len(clean_rows) == 25, f"Expected 25 clean rows, got {len(clean_rows)}"
    checks.append(("clean_rows count (25)", True, ""))
except Exception as e:
    checks.append(("clean_rows count (25)", False, str(e)))

# Check 2: salary is a float in clean rows
try:
    for r in clean_rows:
        assert isinstance(r["salary"], float), f"salary should be float, got {type(r['salary'])} for {r['name']}"
    checks.append(("salary converted to float", True, ""))
except Exception as e:
    checks.append(("salary converted to float", False, str(e)))

# Check 3: no duplicate names in clean_rows
try:
    names = [r["name"] for r in clean_rows]
    assert len(names) == len(set(names)), "Duplicate names found in clean_rows"
    checks.append(("no duplicates in clean_rows", True, ""))
except Exception as e:
    checks.append(("no duplicates in clean_rows", False, str(e)))

# Check 4: age is int or None
try:
    for r in clean_rows:
        assert r["age"] is None or isinstance(r["age"], int), \
            f"age should be int or None, got {type(r['age'])} for {r['name']}"
    checks.append(("age converted to int or None", True, ""))
except Exception as e:
    checks.append(("age converted to int or None", False, str(e)))

# Check 5: employees_clean.csv was written
try:
    assert os.path.exists("employees_clean.csv"), "employees_clean.csv not found"
    with open("employees_clean.csv", "r") as f:
        lines = f.readlines()
    assert len(lines) == 26, f"Expected 26 lines (1 header + 25 data), got {len(lines)}"
    checks.append(("employees_clean.csv written", True, ""))
except Exception as e:
    checks.append(("employees_clean.csv written", False, str(e)))

# Print results
passed = 0
for name, ok, msg in checks:
    icon   = "✓" if ok else "✗"
    status = "PASS" if ok else "FAIL"
    print(f"  {icon} {name}: {status}")
    if msg:
        print(f"      → {msg}")
    if ok:
        passed += 1

print("=" * 45)
print(f"  {passed}/{len(checks)} checks passed")
if passed == len(checks):
    print("  All good — ready to push to GitHub!")
else:
    print("  Fix the issues above and run again.")
print("=" * 45)

  Week 2 Self-check
  ✓ clean_rows count (25): PASS
  ✓ salary converted to float: PASS
  ✓ no duplicates in clean_rows: PASS
  ✓ age converted to int or None: PASS
  ✓ employees_clean.csv written: PASS
  5/5 checks passed
  All good — ready to push to GitHub!
